# Author: Gia Phat Le

GitHub: https://github.com/PhatLene

Email: gple1@myseneca.ca

## Library Import

In [ ]:
import requests
import pandas as pd
import re
import pprint
import matplotlib.pyplot as plt
import seaborn as sns

## Helper Functions

In [ ]:
def iso_to_time(duration):
    h, m, s = re.match(
        r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?',
        duration
    ).groups()
    h, m, s = int(h or 0), int(m or 0), int(s or 0)
    return f"{h}:{m:02}:{s:02}" if h else f"{m}:{s:02}"

def get_qid(claims, prop):
    return claims[prop][0]["mainsnak"]["datavalue"]["value"]["id"] if prop in claims else None

## Wikipedia

In [ ]:
url = "https://en.wikipedia.org/w/api.php"

In [ ]:
dictL = []


pl = ['Lando Norris','Stephen Curry', 'Cristiano Ronaldo', 'Charles Leclerc', 'Giannis Antetokounmpo', 'LaMelo Ball','Taylor Swift', 'Billie Eilish', 'Logan Paul', 'MrBeast']
for person in pl:
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "sites": "enwiki",
        "titles": person,
        "props": "claims",
        "format": "json"
    }
    headers = {"User-Agent": "MyWikiApp/1.0 ldcomia@myseneca.ca"}
    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    entity_id = list(data["entities"].keys())[0]
    claims = data["entities"][entity_id]["claims"]

    dob = claims["P569"][0]["mainsnak"]["datavalue"]["value"]["time"]
    pob = claims["P19"][0]["mainsnak"]["datavalue"]["value"]["id"]
    gender = claims["P21"][0]["mainsnak"]["datavalue"]["value"]["id"]
    citizenship = claims["P27"][0]["mainsnak"]["datavalue"]["value"]["id"]

    qid_map = {
        "place_of_birth": pob,
        "gender": gender,
        "citizenship": citizenship
    }

    ids = "|".join([pob, gender, citizenship])
    params = {
        "action": "wbgetentities",
        "ids": ids,
        "props": "labels",
        "languages": "en",
        "format": "json"
    }
    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    labels_dict = {}
    for prop, qid in qid_map.items():
        entity = data["entities"].get(qid)
        if entity and "labels" in entity and "en" in entity["labels"]:
            labels_dict[prop] = entity["labels"]["en"]["value"]
        else:
            labels_dict[prop] = None
    labels_dict['date_of_birth'] = dob.strip("+").split("T")[0]


    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "titles": person,
        "format": "json"
    }
    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    page = next(iter(data["query"]["pages"].values()))
    intro = page.get("extract", "")
    first_sentence = intro.split(". ")[0] + "." if intro else ""
    labels_dict['Person'] = person
    labels_dict['Wikipeda_FirstSentence'] = first_sentence


    dictL.append(labels_dict)


In [ ]:
df = pd.DataFrame(dictL)
pl = df['Person'].to_list()
df

In [ ]:
sns.countplot(data=df, y='citizenship')
plt.title('Citizenship Distribution')
plt.xlabel('Count')
plt.ylabel('Citizenship')
plt.show()

## Youtube

In [ ]:
API_KEY = "YOUR_YOUTUBE_API_KEY"
channel_url = "https://www.googleapis.com/youtube/v3/channels"
playlist_url = "https://www.googleapis.com/youtube/v3/playlistItems"
videos_url = "https://www.googleapis.com/youtube/v3/videos"

### Obtaining Channel Information

In [ ]:
channel_ids = ['UCwFIQ7wtJzYGFb2nZcVoNww','UC_PXxlBAEnFUXy2Z240jcrQ', 'UCtxD0x6AuNNqdXO9Wp5GHew', 'UCIzIX1qgW7P3eF6JCk_afCA','UCrMiMJMUSjELs0jb1vtTiAg',
               'UCbp_aYD6LNSMQTNYfa5OrwQ','UCqECaJ8Gagnn7YCbPEzWH6g','UCDGmojLIoWpXok597xYo8cg','UCbwMZHCMpBiOWfYLBOH-9Qw','UCX6OQ3DkcsbYNE6H8uQQuVA',]

In [ ]:
## Loop for obtaining channel information including subscriber count, view count, and more
ch_list = []
for ids in channel_ids:
    channel_params = {
    "part": "snippet,statistics,contentDetails,status", ## Determines which data is included
    "id": ids,
    "key": API_KEY}

    channel_data = requests.get(channel_url, params=channel_params).json()

    # Add a check to ensure 'items' key exists and is not empty
    if 'items' not in channel_data or not channel_data['items']:
        print(f"No channel data found for ID: {ids}. Skipping.")
        continue

    ch_description = channel_data['items'][0]['snippet']['description']
    ch_sub_count = channel_data['items'][0]['statistics']['subscriberCount']
    ch_total_vids = channel_data['items'][0]['statistics']['videoCount']
    ch_total_views = channel_data['items'][0]['statistics']['viewCount']
    ch_country = channel_data['items'][0]['snippet'].get('country', 'Unknown') ## Some channels do not provide a country, will be replaced with 'unkown'
    ch_title = channel_data['items'][0]['snippet']['title']
    ch_dict  = {'YT_ChannelTitle': ch_title,
               'YT_SubscriberCount': ch_sub_count,
               'YT_TotalViews': ch_total_views,
               'YT_TotalVideos': ch_total_vids,
               'YT_Country':ch_country,
               'YT_ChannelDescription': ch_description}
    ch_list.append(ch_dict)

In [ ]:
df2 = pd.DataFrame(ch_list)
df2['Person'] = pl
df2.head(10)

In [ ]:
df2['YT_TotalVideos'] = df2['YT_TotalVideos'].astype(int)
plt.figure(figsize=(10, 6))
sns.barplot(data=df2, x='YT_ChannelTitle', y='YT_TotalVideos')
plt.xticks(rotation=45)
plt.xlabel('Channel Title')
plt.ylabel('Total Videos')
plt.title('Total Videos per Channel')
plt.show()


## Twitter

In [ ]:
bearer_token= "YOUR_TWITTER_BEARER_TOKEN"

In [ ]:
users = ['StephenCurry30', 'LandoNorris', 'Cristiano','Charles_Leclerc','Giannis_An34', 'MrBeast', 'billieeilish', 'taylorswift13', 'LoganPaul', 'MELOD1P']

twitterAccs = []
for user in users:
    headers = {"Authorization": f"Bearer {bearer_token}"}
    url = f"https://api.twitter.com/2/users/by/username/{user}"
    params = {"user.fields": "public_metrics"}
    response = requests.get(url, headers=headers, params=params)

    data = response.json()['data']
    acc_id = data['id']

    url_tweets = f"https://api.twitter.com/2/users/{acc_id}/tweets"
    params_tweets = {
        "max_results": 5,
        "tweet.fields": "created_at,public_metrics,text"
    }
    response_tweets = requests.get(url_tweets, headers=headers, params=params_tweets)
    tweets_data = response_tweets.json()
    latest = tweets_data['data'][0]

    twitter_info = {
    "TwitterName": data['name'],
    "TwitterFollowers": data['public_metrics']['followers_count'],
    "NumberOfTweets": data['public_metrics']['tweet_count'],
    "TwitterFollowing": data['public_metrics']['following_count'],
    "TwitterUsername": data['username'], "LatestTweet": latest['text'],
    "LatestTweetTime": latest['created_at'],
    "LatestTweetLikes": latest['public_metrics']['like_count']}

    twitterAccs.append(twitter_info)

In [ ]:
pl2 = ['Stephen Curry', 'Lando Norris','Cristiano Ronaldo', 'Charles Leclerc', 'Giannis Antetokounmpo', 'MrBeast','Billie Eilish', 'Taylor Swift', 'Logan Paul', 'LaMelo Ball']
df3 = pd.DataFrame(twitterAccs)
df3['Person'] = pl2
df3

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=df3, x='TwitterUsername', y='NumberOfTweets')
plt.title('Number of Tweets per Twitter Account')
plt.xticks(rotation=45)
plt.xlabel('Twitter Handle')
plt.ylabel('Number Of Tweets')
plt.show()

## Instagram

In [ ]:
!pip -q install apify-client

In [ ]:
import os
APIFY_TOKEN = os.getenv("APIFY_TOKEN")
APIFY_TOKEN = "YOUR_APIFY_TOKEN"

if not APIFY_TOKEN:
    raise ValueError("Missing APIFY_TOKEN. Add it as an environment variable or set it in the notebook.")

In [ ]:
from apify_client import ApifyClient
import pandas as pd

client = ApifyClient(APIFY_TOKEN)
ig_usernames = [
    'lando',
    'stephencurry30',
    'cristiano',
    'charles_leclerc',
    'giannis_an34',
    "melo",
    "taylorswift",
    "billieeilish",
    "loganpaul",
    "mrbeast",

]

ig_run_input = {
    "usernames": ig_usernames,
    "includeAboutSection": False,  # paid-only per actor docs, so keep False
}

ig_run = client.actor("apify/instagram-profile-scraper").call(run_input=ig_run_input)
ig_items = list(client.dataset(ig_run["defaultDatasetId"]).iterate_items())

# Normalize into a clean dataframe (use .get because actor fields can vary)
ig_rows = []
for it in ig_items:
    ig_rows.append({
        "InstagramUsername": it.get("username") or it.get("userName"),
        "InstagramBio": it.get("biography") or it.get("bio"),
        "InstagramFollowers": it.get("followersCount") or it.get("followers"),
        "InstagramFollowing": it.get("followsCount") or it.get("following"),
        "InstagramPosts": it.get("postsCount") or it.get("posts"),
        "InstagramVerified": it.get("verified"),
        "IG_LatestPostTimestamp": (it.get("latestPosts", [{}])[0].get("timestamp") if isinstance(it.get("latestPosts"), list) and it.get("latestPosts") else None),
        "IG_LatestPostLikes": (it.get("latestPosts", [{}])[0].get("likesCount") if isinstance(it.get("latestPosts"), list) and it.get("latestPosts") else None),
        "IG_LatestPostComments": (it.get("latestPosts", [{}])[0].get("commentsCount") if isinstance(it.get("latestPosts"), list) and it.get("latestPosts") else None),
    })

In [ ]:
ig_df = pd.DataFrame(ig_rows)
name_map = {
    "mrbeast": "MrBeast",
    "cristiano": "Cristiano Ronaldo",
    "melo": "LaMelo Ball",
    "loganpaul": "Logan Paul",
    "billieeilish": "Billie Eilish",
    "giannis_an34": "Giannis Antetokounmpo",
    "stephencurry30": "Stephen Curry",
    "taylorswift": "Taylor Swift",
    "lando": "Lando Norris",
    "charles_leclerc": "Charles Leclerc"
}

ig_df["Person"] = ig_df["InstagramUsername"].map(name_map)
ig_df

In [ ]:
ig_df = ig_df.sort_values(by="InstagramFollowers", ascending=False)

# Instagram Followers Bar Plot

plt.figure(figsize=(10,6))

plt.bar(
    ig_df["Person"],
    ig_df["InstagramFollowers"] / 1_000_000
)

plt.title("Instagram Followers by Public Figure")
plt.xlabel("Public Figures")
plt.ylabel("Followers (Millions)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Instagram Posts Bar Plot
plt.figure(figsize=(10,6))

plt.bar(
    ig_df["Person"],
    ig_df["InstagramPosts"]
)

plt.title("Instagram Posts by Public Figure")
plt.xlabel("Public Figures")
plt.ylabel("Number of Posts")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##Tiktok


In [ ]:
client = ApifyClient(APIFY_TOKEN)
tiktok_usernames = [
    "lamelowithball2",
    "taylorswift",
    "billieeilish",
    "loganpaul",
    "mrbeast",
    'landonorris',


]

tt_run_input = {
    "profiles": tiktok_usernames,     # actor accepts usernames/user IDs
    "resultsPerPage": 5,              # scrape up to 5 recent videos per profile
    "profileScrapeSections": ["videos"],
    "profileSorting": "latest",
    "excludePinnedPosts": False,
}

tt_run = client.actor("clockworks/tiktok-scraper").call(run_input=tt_run_input)
tt_items = list(client.dataset(tt_run["defaultDatasetId"]).iterate_items())

tt_rows = []
for it in tt_items:
    author = it.get("authorMeta") or {}
    stats = it.get("authorStats") or {}
    video_stats = it.get("stats") or {}

    tt_rows.append({
        "Platform": "TikTok",
        "TikTokUsername": author.get("name") or author.get("nickName") or it.get("authorUniqueId"),
        "TikTokDisplayName": author.get("nickName"),
        "TikTokFollowers": stats.get("followerCount"),
        "TikTokFollowing": stats.get("followingCount"),
        "TikTokHearts": stats.get("heartCount"),
        "TikTokVideos": stats.get("videoCount"),
        "TT_LatestVideoTime": it.get("createTimeISO") or it.get("createTime"),
        "TT_LatestVideoLikes": video_stats.get("diggCount"),
        "TT_LatestVideoComments": video_stats.get("commentCount"),
        "TT_LatestVideoShares": video_stats.get("shareCount"),
        "TT_LatestVideoPlays": video_stats.get("playCount"),
        "TT_LatestVideoUrl": it.get("webVideoUrl") or it.get("videoUrl"),
    })

tt_df_raw = pd.DataFrame(tt_rows)
tt_df = (
    tt_df_raw
    .sort_values(by=["TikTokUsername", "TT_LatestVideoTime"], ascending=[True, False])
    .drop_duplicates(subset=["TikTokUsername"], keep="first")
    .reset_index(drop=True)
)

tt_df

# Google Knowledge Graph Search

In [ ]:
import requests
import pandas as pd

GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"
KG_ENDPOINT = "https://kgsearch.googleapis.com/v1/entities:search"

def kg_search_person(query: str, api_key: str, limit: int = 1):
    params = {
        "query": query,
        "key": api_key,
        "limit": limit,
        "indent": True,
        "types": "Person",
        "languages": "en",
    }
    r = requests.get(KG_ENDPOINT, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    items = data.get("itemListElement", [])
    if not items:
        return None

    result = items[0].get("result", {})
    detailed = result.get("detailedDescription", {}) or {}
    image = result.get("image", {}) or {}

    return {
        "KG_Name": result.get("name"),
        "KG_Description": result.get("description"),
        "KG_Types": ", ".join(result.get("@type", [])) if isinstance(result.get("@type"), list) else result.get("@type"),
        "KG_DetailedSummary": detailed.get("articleBody"),
        "KG_DetailedURL": detailed.get("url"),
    }

pl3 = ['MrBeast','Cristiano Ronaldo', 'LaMelo Ball','Logan Paul','Billie Eilish', 'Giannis Antetokounmpo','Stephen Curry','Taylor Swift', 'Lando Norris','Charles Leclerc']

kg_rows = []
for name in pl3:
    row = kg_search_person(name, GOOGLE_API_KEY, limit=1)
    if row is None:
        kg_rows.append({"Platform": "GoogleKnowledgeGraph", "KG_Query": name, "KG_Found": False})
    else:
        kg_rows.append(row)

In [ ]:
kg_df = pd.DataFrame(kg_rows)
kg_df['Person'] = pl3
kg_df

##**Combined Dataframe**

In [ ]:
merge1 = df.merge(df2,on='Person')
merge2 = merge1.merge(df3, on='Person')
merge3 = merge2.merge(kg_df, on='Person')
merge4 = merge3.merge(ig_df, on='Person')
merge4

In [ ]:
merge4.columns

In [ ]:
merge4.to_csv('DatasetOfUnifiedProfiles.csv')

In [ ]:
df_final =pd.read_csv('/content/DatasetOfUnifiedProfiles.csv')
df_final.head(10)